Magesha R.P - Weekly assesment 2

Indian Startup Funding - Decoded

Section A

Q1:A Series is essentially a single column of data with an index, whereas a DataFrame is a 2D table composed of multiple aligned Series sharing the same index. You can use .head() on both to preview data, but an operation like dropping a column via .drop(columns=[...]) is exclusive to DataFrames since a Series doesn't have multiple columns to drop from.

Q2:Setting inplace=True modifies the existing object directly in memory instead of creating a new copy. The pandas team is phasing it out because it doesn't actually save memory, breaks the ability to chain methods together, and frequently causes hidden SettingWithCopyWarning bugs. I’d avoid it in common steps like df.dropna(inplace=True), opting for direct re-assignment (df = df.dropna()) instead.

Q3:.fillna() swaps out missing values (NaN) with a fallback value you choose, while .dropna() just throws away the rows or columns containing those missing values. You’d use .fillna() when dropping rows ruins your dataset size, like filling missing customer age values with the median. You’d pick .dropna() when the missing data is critical and can’t be guessed, like dropping transactions where the customer_id or amount is completely blank.

Q4: Plain pd.to_numeric(col) will immediately crash your script if it hits a single non-numeric piece of text (like "Undisclosed" or "N/A"). Adding errors='coerce' tells pandas to gracefully convert those unparseable strings into NaN flags instead of failing. On messy real-world data, you almost always need errors='coerce' so your data pipeline doesn't break over typos or text entries in currency columns.

Q5:.loc selects data using row and column labels, while .iloc looks strictly at integer positions (0-indexed). For example, df.loc['User_A'] finds the row labeled "User_A", while df.iloc[0] grabs whatever row is sitting at the very top of the table. If you sort your data using df.sort_values(), the physical rows move but retain their original index labels; at that point, df.iloc[0] fetches the new top row, while df.loc[0] looks for the specific row that originally had the number 0 as its label.

Q6:.map() is best for transforming every element using a dictionary look-up or a simple function.

.apply() handles more complex, custom logic across elements via a lambda or a python function.

.replace() is optimized for swapping out specific exact values without touching the rest of the series.
Reach for .replace() to swap out specific typo strings, .map() for key-value dictionary translations, and .apply() for complex math operations.

Q7:It's a warning pandas throws when you try to modify data on a slice of a DataFrame, because it's ambiguous whether you are changing the original dataset or a temporary copy in memory.

Triggers it: df_filtered = df[df['rating'] > 4] followed by df_filtered['rating'] = 5 (chained assignment).

The Fix: Explicitly create an isolated copy when slicing the dataframe by adding .copy(), like: df_filtered = df[df['rating'] > 4].copy().

Q8:pd.cut() sets up bins based on equal-width ranges (e.g., dividing an income scale into $0-\$50k, \$50k-\$100k, \$100k+$), whereas pd.qcut() sets up bins based on sample quantiles to ensure the exact same number of people end up in each bucket. For income tiers, pd.cut() fits much better because economic classes are defined by specific income thresholds, not by forcing exactly 33% of your customers into a "High Earners" group regardless of what they actually make.

Section B

In [1]:
#  Q9
# Predicted Output:
#     a  b
# 0  10  4
# 1  20  5
# 2  30  6
# Explanation: df2 = df creates a reference to the same memory object. Modifying df2 changes df.

import pandas as pd
df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})
df2 = df
df2['a'] = [10, 20, 30]
print(df)

    a  b
0  10  4
1  20  5
2  30  6


In [2]:
# Q10
# Predicted Output:
# 3.0
# 12.0
# Explanation: Pandas statistical functions skip NaN values by default.

import pandas as pd
import numpy as np
s = pd.Series([1, 2, np.nan, 4, 5])
print(s.mean())
print(s.sum())

3.0
12.0


In [3]:
# Q11
# Predicted Output:
# float64
# 700.0
# Explanation: 'NA' is coerced into np.nan which forces the integer column to float precision.

import pandas as pd
df = pd.DataFrame({'price': ['100', '200', 'NA', '400']})
df['price'] = pd.to_numeric(df['price'], errors='coerce')
print(df['price'].dtype)
print(df['price'].sum())

float64
700.0


In [4]:
# Q12
# Predicted Output:
# name    Rahul
# age        22
# Name: 0, dtype: object
# <class 'pandas.core.series.Series'>
# Explanation: Selecting a single row out of a DataFrame yields a Pandas Series object.

import pandas as pd
df = pd.DataFrame({'name': ['Rahul', 'Priya'], 'age': [22, 21]})
print(df.loc[0])
print(type(df.loc[0]))

name    Rahul
age        22
Name: 0, dtype: object
<class 'pandas.core.series.Series'>


In [5]:
# Q13
# Predicted Output:
#    rating
# 0     4.5
# 1     3.8
# 2     4.2
# 3     5.0
#    rating
# 0     9.0
# 2     8.4
# 3    10.0
# Explanation: A SettingWithCopyWarning is raised because df_filtered was assigned from a sliced subset.

import pandas as pd
df = pd.DataFrame({'rating': [4.5, 3.8, 4.2, 5.0]})
df_filtered = df[df['rating'] > 4]
df_filtered['rating'] = df_filtered['rating'] * 2
print(df)
print(df_filtered)

   rating
0     4.5
1     3.8
2     4.2
3     5.0
   rating
0     9.0
2     8.4
3    10.0


/tmp/ipykernel_2711/1226286551.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['rating'] = df_filtered['rating'] * 2


In [ ]:
# Q14
# Predicted Output:
# 0    small
# 1    small
# 2        9
# 3       16
# Name: a, dtype: object
# Explanation: Mixing strings ('small') and integers/floats sets the overall pandas dtype to object.

import pandas as pd
df = pd.DataFrame({'a': [1, 2, 3, 4]})
result = df['a'].apply(lambda x: x**2 if x > 2 else 'small')
print(result)
print(result.dtype)

In [ ]:
# Q15
# Predicted Output:
# 3
# 4
# Delhi        2
# Mumbai       1
# Bengaluru    1
# Name: city, dtype: int64
# Explanation: nunique() excludes NaN. unique() includes NaN, increasing len. value_counts() excludes NaN.

import pandas as pd
df = pd.DataFrame({'city': ['Delhi', 'Mumbai', 'Delhi', 'Bengaluru', None]})
print(df['city'].nunique())
print(len(df['city'].unique()))
print(df['city'].value_counts())

In [ ]:
# Q16
# Predicted Output:
#    a   b
# 1  2  20
# 2  3  30
#    a   b
# 2  3  30
# Explanation: The middle commented line fails because bitwise operations (&) demand parenthesized criteria to resolve operator precedence correctly.

import pandas as pd
df = pd.DataFrame({'a': [1, 2, 3], 'b': [10, 20, 30]})
print(df[df['a'] > 1])
# print(df[df['a'] > 1 & df['b'] > 15])  # Throws ValueError/TypeError
print(df[(df['a'] > 1) & (df['b'] > 15)])

Section C

In [7]:
import pandas as pd
import numpy as np

# Load the raw dataset
df_raw = pd.read_csv("indian_startups_fundingDADS.csv")

# Create copy for manipulation
df = df_raw.copy()

# 1. Rename columns to clean snake_case
df.columns = ['startup_name', 'city', 'industry', 'funding_stage', 'amount', 'funding_date', 'investors', 'founded_year']

# 2. Clean startup_name and city - strip whitespace, title case
df['startup_name'] = df['startup_name'].astype(str).str.strip().str.title()
df['city'] = df['city'].astype(str).str.strip().str.title()

# 3. Standardise city aliases
city_mapping = {
    'Bangalore': 'Bengaluru', 'Blr': 'Bengaluru',
    'Bombay': 'Mumbai',
    'Gurgaon': 'Gurugram',
    'Delhi': 'New Delhi',
    'Calcutta': 'Kolkata',
    'Madras': 'Chennai'
}
df['city'] = df['city'].replace(city_mapping)

# 4. Standardise industry and funding_stage variants using dicts
industry_mapping = {
    'E-Commerce': 'Ecommerce', 'Ecommerce': 'Ecommerce', 'E-commerce': 'Ecommerce',
    'Fin-Tech': 'Fintech', 'Fintech': 'Fintech',
    'Ed-Tech': 'Edtech', 'Edtech': 'Edtech',
    'Health-Tech': 'Health Tech', 'Healthtech': 'Health Tech', 'Health Tech': 'Health Tech'
}
df['industry'] = df['industry'].replace(industry_mapping)

stage_mapping = {
    'Seed Round': 'Seed', 'Seed Funding': 'Seed',
    'Series A1': 'Series A', 'Series A': 'Series A',
    'Pre Series A': 'Pre-Series A', 'Pre-Series A': 'Pre-Series A'
}
df['funding_stage'] = df['funding_stage'].replace(stage_mapping)

# 5. Convert amount to numeric (in INR Cr) without forbidden libraries (No regex used)
def clean_amount(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).replace(',', '').strip()
    if 'Undisclosed' in val_str or val_str == '' or val_str == 'nan':
        return np.nan

    # Check prefixes/suffixes and strip units
    val_str = val_str.replace('INR', '').replace('Cr', '').replace('Crore', '').strip()
    try:
        return float(val_str)
    except:
        return np.nan

df['amount'] = df['amount'].apply(clean_amount)

# 6. Convert funding_date to datetime
df['funding_date'] = pd.to_datetime(df['funding_date'], errors='coerce', dayfirst=True)

# 7. Drop duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

In [8]:
# Q17
print("(a) Rows and columns:", df_raw.shape)
print("(b) Missing values per column:\n", df_raw.isnull().sum())
print("(c) Exact duplicate rows:", df_raw.duplicated().sum())

(a) Rows and columns: (468, 8)
(b) Missing values per column:
 Startup Name      0
City              0
INDUSTRY          0
Funding Stage     0
Amount            0
Funding Date     19
Investors        10
Founded Year     23
dtype: int64
(c) Exact duplicate rows: 18


In [9]:
# Q18
sorted_cols = sorted(list(df.columns))
print(sorted_cols)

['amount', 'city', 'founded_year', 'funding_date', 'funding_stage', 'industry', 'investors', 'startup_name']


In [10]:
# Q19
founded_clean = df['founded_year'].dropna()
stats_dict = {
    'count': int(founded_clean.count()),
    'mean': float(founded_clean.mean()),
    'min': float(founded_clean.min()),
    'max': float(founded_clean.max()),
    'median': float(founded_clean.median()),
    'std': float(founded_clean.std())
}
print(stats_dict)

{'count': 427, 'mean': 2016.4285714285713, 'min': 2008.0, 'max': 2022.0, 'median': 2017.0, 'std': 4.246117096325488}


In [11]:
# Q20
unique_cities = sorted(df['city'].dropna().unique())
print("Unique Cities List:", unique_cities)
print("Count of Unique Cities:", len(unique_cities))

Unique Cities List: ['Ahmedabad', 'Bengaluru', 'Bom', 'Chennai', 'Ggn', 'Gurugram', 'Hyd', 'Hyderabad', 'Kolkata', 'Mumbai', 'New Delhi', 'Noida', 'Pune']
Count of Unique Cities: 13


In [12]:
#Q21
print(df['industry'].value_counts())

industry
AI/ML                22
Ecommerce            18
TRAVELTECH           16
saas                 14
logistics            14
BEAUTYTECH           13
Health Tech          13
Travel-Tech          13
PROPTECH             13
BeautyTech           12
Agri Tech            12
Electric Vehicles    11
Prop-Tech            11
AIML                 11
GAMING               11
AI-ML                11
Edtech               11
AI ML                10
Saas                 10
FoodTech             10
Gaming                9
Food-Tech             9
Beauty-Tech           9
ecommerce             8
Food Tech             8
Games                 8
PropTech              8
LOGISTICS             8
Fintech               8
HealthTech            7
foodtech              7
EdTech                7
EV                    6
SAAS                  6
SaaS                  6
healthtech            6
gaming                6
TravelTech            5
FINTECH               5
Electric-Vehicles     5
Logistics             5
ev     

In [13]:
# Q22
print("(a) Dtype:", df['amount'].dtype)
print("(b) NaN count (from Undisclosed entries):", df['amount'].isna().sum())
print("(c) Mean funding amount:", df['amount'].mean())
print("(d) Min amount:", df['amount'].min(), "| Max amount:", df['amount'].max())

(a) Dtype: float64
(b) NaN count (from Undisclosed entries): 281
(c) Mean funding amount: 761.2781065088758
(d) Min amount: 8.0 | Max amount: 3545.0


In [14]:
# Q23
deals_per_year = df['funding_date'].dt.year.value_counts()
print("(a) Deals per year:\n", deals_per_year)
print("(b) Year with highest deals:", deals_per_year.idxmax())

(a) Deals per year:
 funding_date
2022.0    23
2023.0    14
2021.0    14
2024.0     7
2019.0     7
2020.0     6
2017.0     5
2018.0     5
2016.0     4
2015.0     3
2011.0     1
2014.0     1
2013.0     1
2012.0     1
Name: count, dtype: int64
(b) Year with highest deals: 2022.0


In [15]:
# Q24
print(df['funding_stage'].value_counts())

funding_stage
Seed            21
Pre-Series A    20
ipo             18
Series-E        17
bridge          16
IPO             15
Bridge          15
SERIES A        14
series_b        14
SERIES C        14
Pre-Seed        14
SEED            13
seed            13
SERIES E        13
Pre Seed        12
series b        12
Series-D        12
Series-B        12
Pre-IPO         12
Series-C        12
PRE-SEED        12
Series E        11
series d        11
Series-A        10
Series D        10
Series C        10
series c        10
BRIDGE          10
Preseed         10
Pre-A            9
Series B         9
SERIES B         9
series_a         9
PRE-SERIES A     9
Series A         7
series e         7
SERIES D         5
series a         3
Name: count, dtype: int64


In [16]:
# Q25
blr_large = df[(df['city'] == 'Bengaluru') & (df['amount'] > 100)].sort_values(by='amount', ascending=False)
print(blr_large[['startup_name', 'amount']])

       startup_name  amount
301        Routemap  3528.0
223     Agriconnect  3526.0
152        Ecomotor  2244.0
98           Meesho  2231.0
144       Curefoods  2230.0
247  Playtime Games  1410.0
355            Cred   950.0
67      Simplilearn   923.0
431        Agroplus   726.0
330          Practo   486.0
88            Zepto   455.0
147     Mediconnect   304.0
180         Fetchit   150.0
384         Cuemath   115.0


In [17]:
# Q26
target_industries = ['Fintech', 'Edtech', 'Health Tech']
count_target = df[df['industry'].isin(target_industries)].shape[0]
print("Total count:", count_target)

Total count: 32


In [18]:
# Q27
founded_between = df[df['founded_year'].between(2015, 2020)].shape[0]
print("Startups founded between 2015 and 2020 inclusive:", founded_between)

Startups founded between 2015 and 2020 inclusive: 211


In [19]:
# Q28
pay_startups = df[df['startup_name'].str.contains('pay', case=False, na=False)]['startup_name'].unique()
print(pay_startups)

['Rupaypro' 'Razorpay' 'Paytm' 'Paybuddy' 'Quickpay India']


In [20]:
# Q29
df['funding_efficiency'] = df['amount'] / (2024 - df['founded_year'])
top_10_efficient = df.sort_values(by='funding_efficiency', ascending=False).head(10)
print(top_10_efficient[['startup_name', 'industry', 'amount', 'founded_year', 'funding_efficiency']])

    startup_name           industry  amount  founded_year  funding_efficiency
311   Classroomx              AI/ML  3520.0        2022.0              1760.0
152     Ecomotor             gaming  2244.0        2022.0              1122.0
156   Makemytrip  Electric Vehicles  2202.0        2022.0              1101.0
55       Blinkit  Electric Vehicles  2196.0        2022.0              1098.0
165  Fitnessplus              AI/ML  1430.0        2022.0               715.0
28      Rupaypro        Travel-Tech  3541.0        2019.0               708.2
202    Docontime         TRAVELTECH  3529.0        2019.0               705.8
142    Groomguru             Gaming  3497.0        2019.0               699.4
368    Agriboost              Games  3507.0        2018.0               584.5
144    Curefoods          Food-Tech  2230.0        2020.0               557.5


In [21]:
# Q30
def get_tier(amt):
    if pd.isna(amt):
        return 'Small / Unknown'
    elif amt >= 1000:
        return 'Unicorn'
    elif amt >= 500:
        return 'Mega'
    elif amt >= 100:
        return 'Large'
    elif amt >= 10:
        return 'Mid'
    else:
        return 'Small / Unknown'

df['funding_tier'] = df['amount'].apply(get_tier)
print(df['funding_tier'].value_counts())

funding_tier
Small / Unknown    282
Large               53
Mid                 50
Unicorn             38
Mega                27
Name: count, dtype: int64


In [22]:
# Q31
# Using manual bin assignments matching pd.cut criteria boundary logic safely
df['era'] = pd.cut(df['founded_year'], bins=[-np.inf, 2009, 2017, np.inf], labels=['Early', 'Growth', 'Recent'])
print(df['era'].value_counts())

era
Recent    207
Growth    195
Early      25
Name: count, dtype: int64


In [23]:
# Q32
df['investor_count'] = df['investors'].apply(lambda x: len(str(x).split(', ')) if pd.notna(x) and str(x) != 'nan' else 0)
print(df['investor_count'].value_counts())

investor_count
2    199
3    118
1     94
4     29
0     10
Name: count, dtype: int64


In [24]:
# Q33
avg_funding_industry = df.groupby('industry')['amount'].mean().sort_values(ascending=False)
print(avg_funding_industry.head(5))

industry
Fintech       2863.500000
TRAVELTECH    1609.600000
Food-Tech     1500.250000
Logistics     1482.333333
HealthTech    1421.000000
Name: amount, dtype: float64


In [25]:
# Q34
city_summary = df.groupby('city').agg(
    startup_count=('startup_name', 'count'),
    total_funding=('amount', 'sum'),
    avg_funding=('amount', 'mean')
)
print(city_summary.sort_values(by='startup_count', ascending=False).head(10))

           startup_count  total_funding  avg_funding
city                                                
Noida                 62        15500.0   673.913043
Kolkata               51        12786.0   639.300000
Bengaluru             49        19525.0   929.761905
Ahmedabad             44        18372.0   966.947368
Pune                  42        12797.0   710.944444
Chennai               37         9750.0   609.375000
Gurugram              36        14136.0   831.529412
Mumbai                35         7802.0   709.272727
New Delhi             33         8626.0   958.444444
Hyderabad             30         7247.0   805.222222


In [27]:
# Q35
ind_group = df.groupby('industry').agg(
    startup_count=('startup_name', 'count'),
    avg_funding=('amount', 'mean')
)
filtered_industries = ind_group[ind_group['startup_count'] >= 15]
highest_dense_industry = filtered_industries.sort_values(by='avg_funding', ascending=False).head(1)
print(highest_dense_industry)

            startup_count  avg_funding
industry                              
TRAVELTECH             16       1609.6


In [28]:
# Q36
top_5_industries = df['industry'].value_counts().head(5).index
df_top_ind = df[df['industry'].isin(top_5_industries)]
cross_tab = pd.crosstab(df_top_ind['industry'], df_top_ind['funding_tier'])
print(cross_tab)

funding_tier  Large  Mega  Mid  Small / Unknown  Unicorn
industry                                                
AI/ML             3     1    3               12        3
Ecommerce         1     0    2               15        0
TRAVELTECH        0     3    0               11        2
logistics         2     1    2                9        0
saas              2     1    1               10        0


In [29]:
# Q37
# Drop missing amounts to avoid incorrect idxmax metrics
df_amount_clean = df.dropna(subset=['amount'])
idx_max = df_amount_clean.groupby('industry')['amount'].idxmax()
highest_funded_per_ind = df_amount_clean.loc[idx_max, ['startup_name', 'industry', 'city', 'amount']]
print(highest_funded_per_ind)

      startup_name           industry       city  amount
252       Edusmart           AGRITECH      Noida   193.0
124  Bookmyservice              AI ML    Chennai  1443.0
392         Glowup              AI-ML     Mumbai  1441.0
311     Classroomx              AI/ML  Ahmedabad  3520.0
328      Sharechat               AIML   Gurugram   900.0
390      Beautybox          Agri Tech   Gurugram   183.0
2       Freshworks           AgriTech    Kolkata   329.0
98          Meesho         BEAUTYTECH  Bengaluru  2231.0
93       Vitalcare         BeautyTech       Pune  3522.0
261    Whitehat Jr          ECOMMERCE    Kolkata    98.0
305      Instaloan                 EV    Chennai   923.0
9      Mediconnect          Ecommerce      Noida   295.0
444        Postman             EdTech  Hyderabad  1404.0
173       Propmart             Edtech  New Delhi  3534.0
301       Routemap  Electric Vehicles  Bengaluru  3528.0
12             Ola  Electric-Vehicles  New Delhi  3520.0
273      Greengrow          FOO

In [30]:
# Q38
emerging_players = df[
    (df['founded_year'] >= 2019) &
    (df['amount'] >= 100) &
    (df['industry'].isin(['Fintech', 'Edtech', 'Health Tech']))
].sort_values(by='amount', ascending=False)

print(emerging_players[['startup_name', 'industry', 'city', 'founded_year', 'amount']])

# Analysis insight comment:
# A VC analyst cares deeply about this list because it identifies young market disruptors that
# have achieved massive capital traction very rapidly, representing strong growth-signals.

    startup_name     industry      city  founded_year  amount
331      Voltcar       Edtech       Hyd        2021.0   916.0
402  Villagemart       Edtech  Gurugram        2021.0   896.0
128    Dermaplus  Health Tech  Gurugram        2019.0   711.0


In [31]:
# Q39
city_hubs = df.groupby('city').agg(
    startup_count=('startup_name', 'count'),
    total_funding=('amount', 'sum')
)
filtered_hubs = city_hubs[city_hubs['startup_count'] >= 20].sort_values(by='total_funding', ascending=False)
print(filtered_hubs)

           startup_count  total_funding
city                                   
Bengaluru             49        19525.0
Ahmedabad             44        18372.0
Noida                 62        15500.0
Gurugram              36        14136.0
Pune                  42        12797.0
Kolkata               51        12786.0
Chennai               37         9750.0
New Delhi             33         8626.0
Mumbai                35         7802.0
Hyderabad             30         7247.0


In [32]:
# Q40
saturation = df.groupby(['city', 'industry']).size().sort_values(ascending=False)
print("Top 10 Most-Saturated Combinations:\n", saturation.head(10))

Top 10 Most-Saturated Combinations:
 city       industry         
Noida      BEAUTYTECH           6
           FoodTech             5
Pune       AI/ML                4
New Delhi  LOGISTICS            4
Noida      TRAVELTECH           4
Kolkata    AI/ML                4
Noida      PROPTECH             3
Ahmedabad  Travel-Tech          3
Noida      logistics            3
           Electric Vehicles    3
dtype: int64
